# 08V · T-cell feature/QC tests and adaptive gene weighting

**Project:** Adagrasib_mouse_project_VisiumHD_2026  
**Release:** 1.0.1 · 18 November 2026 · adaptive GPU startup fix  
**Question:** Do rare-feature filtering, removal of variance scaling, or a bounded depth-aware adaptive weighting step improve the existing T cell 1 + T cell 2 analysis?

## Start here

1. Put this notebook beside the working **08T/08U notebook**.
2. Select the **same UV/RAPIDS GPU kernel**, then **Run All**. The default run includes every main comparison below.
3. Usually the only setting you might need to edit is `PROJECT_DIR` in the next cell. It is the original code/project folder, not an H5AD file.
4. At the end, open the review and return **08V_return_to_chat.zip**. The return bundle is capped at 25 MB.

**If your previous run stopped at “Adaptive CUDA parity probe timed out” after 60 seconds:** choose **Restart Kernel and Run All** in this version. Keep your existing result folders. The repaired startup check runs before input preparation or model fitting.

The notebook embeds its helpers and uses the completed 08O source manifest plus compatible existing caches. It creates a separate **21_tcell_feature_qc_dynamic_weights** folder. No SAM installation is needed. No new physical-neighbor feature construction is run.

### Five matched main models, plus one separate QC refit

| Model | Genes | Metric before PCA | Purpose |
|---|---|---|---|
| `selected_zscore` | Frozen 08U reference panel | Centering and unit variance | Reproduce the feature/scaling choice implicated in tiny islands |
| `selected_centered` | Exactly the same panel | Centering only | Isolate variance scaling |
| `selected500_centered` | Same panel intersected with ≥500 detected cells | Centering only | Isolate very rare features without selecting replacement genes |
| `broad_ratio` | Broad genes detected in ≥500 cells | Original ratio weights, midpoint 0.60 | Matched static baseline for adaptation |
| `broad_adaptive` | Exactly the same broad pool | Ratio prior plus at most two bounded updates | Test neighborhood coherence and a depth penalty |
| `qc_ratio` | Same broad pool, frozen before QC | Same ratio weights | Separate refit after counts ≥200 and detected common genes ≥100 |

Every main model uses **all original T-cell rows**. The QC refit is explicitly a different cohort; its plots compare the same retained cells against their full-cohort fit. QC masks at **100 counts/50 common genes** and **200 counts/100 common genes** are both audited by sample and condition. The second mask is the default refit. These are sensitivity tests, not a change to the atlas QC or annotation.

**Held-out genes:** `Cd8b1`, `Il2ra`, `Ctla4`, `Ikzf2`, `Tnfrsf18` are removed before the first embedding in every arm. Their original assays remain available for review. Thus the selected-gene control is the 08U reference panel with these reserved companions removed, not a byte-for-byte rerun of the original 08U main model.

In [ ]:
from pathlib import Path

# Leave this unchanged first. If automatic discovery fails, enter the ORIGINAL
# project folder containing ct26_analysis_config.json and inputs/pipeline_config.json.
PROJECT_DIR = Path.cwd()

### Embedded helpers

The collapsed cell restores the exact helper sources included in this notebook. It downloads nothing and does not install or upgrade packages. A readable source package is also available. Use a fresh kernel if a previous GPU analysis is still holding substantial memory.

In [ ]:
import base64, hashlib, importlib, json, os, shutil, sys, time, traceback, zipfile, zlib

_source_bytes = zlib.decompress(base64.b64decode('...'))
HELPER_SOURCE_ID = '9a87d4e8159f11b2a2300310f084d7f0784938b73a6eb7631cfe31efb63cd226'
if hashlib.sha256(_source_bytes).hexdigest() != HELPER_SOURCE_ID:
    raise RuntimeError("Embedded helper integrity check failed; download the notebook again.")
_sources = json.loads(_source_bytes)
HELPER_DIR = Path.cwd() / ".08V_helpers" / HELPER_SOURCE_ID[:16]
HELPER_DIR.mkdir(parents=True, exist_ok=True)
for _name, _source in _sources.items():
    _path = HELPER_DIR / _name
    if not _path.is_file() or _path.read_text(encoding="utf-8") != _source:
        _path.write_text(_source, encoding="utf-8")
    sys.modules.pop(Path(_name).stem, None)
sys.path.insert(0, str(HELPER_DIR))
importlib.invalidate_caches()
for _variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_variable, "8")

import numpy as np
import pandas as pd
from IPython.display import display, FileLink, HTML, Image
import tcell_adaptive_workflow as workflow
import tcell_adaptive_runtime as runtime
import tcell_adaptive_reports as reports
import tcell_weighted_gpu as gpu

EARLY_DIAGNOSTICS = Path.cwd() / "08V_diagnostics" / HELPER_SOURCE_ID[:16]
EARLY_DIAGNOSTICS.mkdir(parents=True, exist_ok=True)
NOTEBOOK_STARTED = time.monotonic()

def save_failure(exc, plan=None, run_root=None):
    trace = traceback.format_exc()
    (EARLY_DIAGNOSTICS / "notebook_error.txt").write_text(trace, encoding="utf-8")
    if plan is not None:
        try:
            bundle = reports.make_return_bundle(plan, run_root=run_root, error=exc)
            local_bundle = Path.cwd() / "08V_failure_diagnostics.zip"
            if bundle.resolve() != local_bundle.resolve():
                shutil.copy2(bundle, local_bundle)
            display(FileLink(os.path.relpath(local_bundle, Path.cwd())))
            return local_bundle
        except Exception as bundle_error:
            print("Could not build the full return bundle:", bundle_error)
    bundle = Path.cwd() / "08V_failure_diagnostics.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in EARLY_DIAGNOSTICS.rglob("*"):
            if path.is_file() and path.suffix in (".json", ".txt", ".log") and path.stat().st_size < 2000000:
                archive.write(path, str(path.relative_to(EARLY_DIAGNOSTICS)))
    print("Return this diagnostic file:", bundle)
    display(FileLink(os.path.relpath(bundle, Path.cwd())))
    return bundle

print("08V 1.0.1 · helpers restored:", HELPER_SOURCE_ID[:16])
print("Kernel:", sys.executable)

### Defaults for the first run

No choices are required here. Advanced settings are collected in the collapsed cell so the first run remains reproducible. GPU work uses up to four visible GPUs, one complete model per worker. The adaptive model waits for its static baseline. It is not a multi-GPU fit of a single model.

The **GPU stage** has a 110-minute budget. Input preparation and report generation are additional. This is a checkpointed budget, not a promise of a two-hour end-to-end completion time. Rerunning reuses compatible completed stages.

In [ ]:
PREP_SETTINGS = {
    "midpoint": 0.60, "power": 2.0, "target_sum": 10000.0,
    "selected_n_top_genes": 3000, "selected_min_detected_cells": 20,
    "broad_min_detected_cells": 500,
    "run_qc_branch": True, "qc_min_counts": 200, "qc_min_genes": 100,
    "qc_min_cells": 1000, "resume": True,
}
GPU_SETTINGS = dict(gpu.DEFAULT_SETTINGS)
ADAPTIVE_SETTINGS = dict(runtime.ADAPTIVE_DEFAULTS)
ADAPTIVE_PREFLIGHT_TIMEOUT_SECONDS = 600  # startup check only; defaults need no edits
MAX_GPU_STAGE_MINUTES = 110
PRIMARY_CLUSTER_KEY = "leiden_0.4"
MAX_PLOT_CELLS = 100000  # display only; marker statistics use all declared cells
RESUME = True

## 1. Confirm GPU access and installed APIs

This uses fresh subprocesses so the notebook kernel does not initialize CUDA. After the existing PCA/Harmony/neighbor/Leiden/UMAP test, the adaptive check compares three CPU and GPU results on a tiny synthetic fixture. It now allows **10 minutes**, shows its current stage and emits a heartbeat about every 15 seconds while waiting. A real numerical mismatch still stops the run. If startup fails again, return the new **08V_failure_diagnostics.zip** with the stage log. Both startup checks are separate from the 110-minute model budget; the notebook respects the GPUs visible to this kernel.

In [ ]:
try:
    GPU_RUNTIME = runtime.preflight(HELPER_DIR, EARLY_DIAGNOSTICS / "preflight",
                                    adaptive_timeout_seconds=ADAPTIVE_PREFLIGHT_TIMEOUT_SECONDS)
    _visible = GPU_RUNTIME.get("visible_device_count", GPU_RUNTIME.get("device_count"))
    print("CUDA/RAPIDS and adaptive scorer ready; visible GPUs:", _visible)
    if _visible == 1:
        print("One GPU is visible. This is supported; comparison models will run one at a time.")
except (Exception, KeyboardInterrupt) as exc:
    save_failure(exc)
    raise

## 2. Reuse the original T cells and audit features/QC

The source receipt fixes the cells and their order. A matching 08U selection is reused when available; otherwise that reference selection is reproduced from the original counts. The environmental-ratio prior is inherited from the existing workflow.

Normalization always uses each cell's **original full-source library size**. Filtering features never renormalizes the denominator. `n_genes` in this workflow counts detected genes in the **08O common-gene universe**, so the QC table names it accordingly. Original count matrices and atlas labels are unchanged.

In [ ]:
PLAN = None
try:
    PLAN = workflow.prepare_inputs(PROJECT_DIR, settings=PREP_SETTINGS)
    panel_rows = []
    for name, info in PLAN["branches"].items():
        features = pd.read_csv(Path(info["cache_dir"]) / "features.csv")
        manifest = json.loads((Path(info["cache_dir"]) / "cache_manifest.json").read_text())
        panel_rows.append({"model": name, "cohort": info["cohort"],
                           "genes": len(features), "positive_weight_genes": int((features.weight > 0).sum()),
                           "description": info["label"]})
    display(pd.DataFrame(panel_rows))
    QC_RETENTION = pd.read_csv(PLAN["qc"]["retention_path"])
    display(QC_RETENTION[QC_RETENTION.scope.isin(["all", "condition"])])
    print("Plan:", Path(PLAN["plan_dir"]) / "plan.json")
except Exception as exc:
    save_failure(exc, plan=PLAN)
    raise

## 3. Run GPU models and adaptive updates

### What the adaptive model learns

For gene (g), the existing environment-to-T ratio (r_g) supplies the positive prior

\[
q_g = \frac{1}{1+(r_g/0.60)^2}.
\]

Within each sample, the scorer fits an intercept plus linear and quadratic log library size. It records the fraction of within-sample gene variance associated with those depth terms, (D_g\in[0,1]). Residual expression is used **only to learn weights**.

The reward (H_g\in[0,1]) measures agreement between a cell's residual expression and the mean residual expression of its expression-graph neighbors. Self edges are removed. Correlations are averaged across supported samples, truncated below zero, normalized against the eligible 95th percentile, clipped, and shrunk by `detected_cells / (detected_cells + 500)`.

\[
w_g^* = q_g\exp\{\log(3)H_g-\log(3)D_g\},\qquad
\log w_g^{new}=0.5\log w_g^{old}+0.5\log w_g^*.
\]

The target stays between (q_g/3) and (3q_g). The default makes at most **two updates**, with early stopping only if the 95th percentile of absolute log-weight changes is ≤0.01. Each update starts from the fixed ratio prior in its target, rather than multiplying rewards without a bound.

The model multiplies centered original log-normalized expression by **\(\sqrt{w_g}\)** before PCA. No variance scaling is applied afterward in either broad-gene arm. The score estimates use up to **6,000 reproducible probe cells per sample**, at most **72,000 overall**. Depth fits, neighborhood donors, PCA, Harmony, clustering and final UMAP retain **every cell** in the declared cohort.

**This is SAM-inspired, not an implementation or reproduction of SAM.** No named CD4/CD8/Foxp3 marker or predicted cell type receives a manual positive reward. Coherence is a proxy for structured signal, not proof of biology. Depth association can include real activation or cell-size effects. Reserved companion genes do not participate in any initial or updated embedding.

In [ ]:
RUN_ROOT = None
try:
    BRANCHES, RUN_ROOT = runtime.launch_experiment(
        PLAN, HELPER_DIR, GPU_RUNTIME,
        gpu_settings=GPU_SETTINGS, adaptive_settings=ADAPTIVE_SETTINGS,
        max_minutes=MAX_GPU_STAGE_MINUTES, resume=RESUME,
    )
    display(pd.DataFrame([{"model": name, "embedding": str(path)} for name, path in BRANCHES.items()]))
except Exception as exc:
    RUN_ROOT = getattr(exc, "run_root", RUN_ROOT)
    save_failure(exc, plan=PLAN, run_root=RUN_ROOT)
    raise

## 4. Review original markers, depth, fragmentation and held-out companions

Reports include CD4/CD8/Treg lineage markers, naive/memory, cytotoxic, activation/dysfunction and cycling programs, plus NK and environmental markers. Dot size shows the observed detection fraction; color shows original log-normalized expression, including zeros. Condition panels are included.

The static/adaptive comparison is the main test of the learned weights. The feature/scaling comparison and separate QC refit help explain anomalies. Look for agreement across original-marker patterns, reserved companion evidence and sample/condition panels. A cleaner UMAP or greater cluster purity alone does not establish a better solution.

Held-out neighbor statistics account for coarse sample/depth strata and exclude self edges. These are internal feature checks; they are not cell-type accuracy estimates or independent validation of CD4 identity. Repeatedly choosing settings using the same held-out markers also turns them into tuning evidence.

In [ ]:
try:
    REVIEW = reports.build_reports(
        PLAN, BRANCHES, RUN_ROOT,
        primary_cluster_key=PRIMARY_CLUSTER_KEY,
        cluster_keys=[f"leiden_{float(value):g}" for value in GPU_SETTINGS["resolutions"]],
        max_plot_cells=MAX_PLOT_CELLS,
    )
    DISPLAY_DIR = Path.cwd() / "08V_results" / PLAN["plan_id"][:12] / Path(RUN_ROOT).name
    shutil.copytree(Path(REVIEW["index_path"]).parent, DISPLAY_DIR, dirs_exist_ok=True)
    REVIEW_DIR = DISPLAY_DIR
    COMPARISON = pd.read_csv(REVIEW["comparison_path"])
    display(COMPARISON[COMPARISON.clustering == PRIMARY_CLUSTER_KEY])
    display(FileLink(os.path.relpath(DISPLAY_DIR / "index.html", Path.cwd()),
                     result_html_prefix="Open the full review: "))
except Exception as exc:
    save_failure(exc, plan=PLAN, run_root=RUN_ROOT)
    raise

In [ ]:
for relative in [
    "adaptive/figures/umap_lineage_markers.png",
    "adaptive/companions/figures/heldout_companion_concordance.png",
    "weights/adaptive_weight_diagnostics.png",
    "feature_scaling/figures/umap_clusters.png",
]:
    figure = REVIEW_DIR / relative
    if figure.is_file():
        display(HTML(f"<h3>{relative}</h3>"))
        display(Image(filename=str(figure), width=1150))

## 5. Save the small return-to-chat package

The complete reports and models remain in the run folder. The return ZIP contains the key figures, original-marker summary tables, QC retention, learned gene weights and provenance. It excludes large expression/graph arrays and per-cell exports. If needed, lower-priority figures are omitted to keep the bundle below 25 MB; omissions are listed in its manifest.

In [ ]:
LOCAL_RETURN = DISPLAY_DIR / "08V_return_to_chat.zip"
reports.make_return_bundle(PLAN, RUN_ROOT, destination=LOCAL_RETURN)
display(FileLink(os.path.relpath(LOCAL_RETURN, Path.cwd()),
                 result_html_prefix="Return this small ZIP: "))
print(f"Notebook elapsed: {(time.monotonic() - NOTEBOOK_STARTED) / 60:.1f} min")
print("Full results:", RUN_ROOT)
print("Please return:", LOCAL_RETURN)

## Interpretation and next decision

- **Scaling:** compare `selected_zscore` with `selected_centered`. A change here supports a scaling effect, not a new cell identity.
- **Rare genes:** compare `selected_centered` with `selected500_centered`. This removes rare features from the same frozen panel; it does not replace them with a different HVG selection.
- **Adaptive value:** compare `broad_ratio` with `broad_adaptive`. Their original cells, feature pool, normalization and downstream settings match.
- **QC:** inspect retained fractions by condition before interpreting the refit. The QC review compares identical retained rows, and avoids a misleading comparison between an induced old graph and a newly fitted graph.
- **Biological support:** inspect Cd4, Cd8a, the held-out Cd8b1 assay, Foxp3 and reserved Treg companions together. Observed zero counts do not establish biological absence. A single Foxp3-positive observation is not a complete Treg annotation.

A useful adaptive result should improve coherent original-marker and reserved-companion patterns without creating a new depth/sample partition or erasing treatment-associated structure. There is no automatic winner. The outputs are gene-space distance weights and embeddings, **not corrected counts** for a count-likelihood model such as scVI.

### Sources and implementation provenance

- Tarashansky et al. (2019), *Self-assembling manifolds in single-cell RNA sequencing data*, eLife 8:e48994: https://pmc.ncbi.nlm.nih.gov/articles/PMC6795480/
- SAM repository: https://github.com/atarashansky/self-assembling-manifold
- Source inspected at commit `28c16607d469d5a9954bea7649ac50bbc16ca0a6`: https://github.com/atarashansky/self-assembling-manifold/blob/28c16607d469d5a9954bea7649ac50bbc16ca0a6/src/samalg/sam.py
- RAPIDS SingleCell Harmony API: https://rapids-singlecell.scverse.org/en/stable/api/generated/rapids_singlecell.pp.harmony_integrate.html

SAM learns weights from neighborhood-averaged expression and repeatedly reconstructs a manifold. The present pilot substitutes depth-residual neighborhood correlation, a pre-existing ratio prior, explicit bounds and damping. The scientific assumption is exploratory. The source package documents the exact tests; no actual CUDA run or biological improvement is claimed before running this notebook on the EC2 data.